Este notebook genera datos de prueba a partir de samples.nyctaxi.trips, no contiene ninguna lógica del pipeline ETL

In [0]:
# ============================================
# CONFIGURACIÓN DE LA SIMULACIÓN
# ============================================

NUM_CARGAS = 4
REGISTROS_POR_CARGA = 5000

PORCENTAJE_DUPLICADOS = 0.05
PORCENTAJE_ERRORES = 0.02

SEED = 42

FUENTE = "samples.nyctaxi.trips"

print("Configuración de simulación")
print(f"Número de cargas: {NUM_CARGAS}")
print(f"Registros por carga: {REGISTROS_POR_CARGA}")
print(f"Porcentaje duplicados: {PORCENTAJE_DUPLICADOS:.0%}")
print(f"Porcentaje errores: {PORCENTAJE_ERRORES:.0%}")
print(f"Semilla aleatoria: {SEED}")

In [0]:
# ============================================
# DATASET BASE
# ============================================

df_base = spark.table("samples.nyctaxi.trips")

print(f"Registros disponibles: {df_base.count()}")

df_base.printSchema()

In [0]:
# ============================================
# PREPARAR DATASET PARA SIMULACIÓN
# ============================================

from pyspark.sql import functions as F

df_base = df_base.orderBy(F.rand(SEED))

df_simulacion = df_base.limit(
    NUM_CARGAS * REGISTROS_POR_CARGA
)

print(f"Registros seleccionados: {df_simulacion.count()}")

Creacion de las cargas simuladas

In [0]:
from pyspark.sql.window import Window

# ============================================
# ASIGNAR REGISTROS A CARGAS
# ============================================

window_simulacion = Window.orderBy(
    F.monotonically_increasing_id()
)

df_simulacion = (
    df_simulacion
    .withColumn(
        "_id_simulacion",
        F.row_number().over(window_simulacion)
    )
    .withColumn(
        "_carga_base",
        F.ceil(F.col("_id_simulacion") / REGISTROS_POR_CARGA)
    )
)

df_simulacion.groupBy("_carga_base").count().orderBy("_carga_base").display()

In [0]:
# ============================================
# SEPARAR LAS CARGAS BASE
# ============================================

carga_001 = (
    df_simulacion
    .filter(F.col("_carga_base") == 1)
    .drop("_id_simulacion", "_carga_base")
)

carga_002 = (
    df_simulacion
    .filter(F.col("_carga_base") == 2)
    .drop("_id_simulacion", "_carga_base")
)

carga_003 = (
    df_simulacion
    .filter(F.col("_carga_base") == 3)
    .drop("_id_simulacion", "_carga_base")
)

carga_004 = (
    df_simulacion
    .filter(F.col("_carga_base") == 4)
    .drop("_id_simulacion", "_carga_base")
)

print(f"Carga 001: {carga_001.count()} registros")
print(f"Carga 002: {carga_002.count()} registros")
print(f"Carga 003: {carga_003.count()} registros")
print(f"Carga 004: {carga_004.count()} registros")

Creacion de duplicados dentro de la misma carga

In [0]:
# ============================================
# GENERAR DUPLICADOS INTERNOS
# ============================================

def agregar_duplicados_internos(df, porcentaje, seed):

    cantidad = int(df.count() * porcentaje)

    duplicados = (
        df
        .sample(
            withReplacement=False,
            fraction=porcentaje,
            seed=seed
        )
        .limit(cantidad)
    )

    return df.unionByName(duplicados)


carga_001 = agregar_duplicados_internos(
    carga_001,
    PORCENTAJE_DUPLICADOS,
    SEED + 1
)

carga_002 = agregar_duplicados_internos(
    carga_002,
    PORCENTAJE_DUPLICADOS,
    SEED + 2
)

carga_003 = agregar_duplicados_internos(
    carga_003,
    PORCENTAJE_DUPLICADOS,
    SEED + 3
)

carga_004 = agregar_duplicados_internos(
    carga_004,
    PORCENTAJE_DUPLICADOS,
    SEED + 4
)

print(f"Carga 001: {carga_001.count()} registros")
print(f"Carga 002: {carga_002.count()} registros")
print(f"Carga 003: {carga_003.count()} registros")
print(f"Carga 004: {carga_004.count()} registros")

creacion de duplicados entre cargas diferentes

In [0]:
# ============================================
# GENERAR DUPLICADOS HISTÓRICOS
# ============================================

def obtener_duplicados_historicos(df_historico, cantidad, seed):

    total = df_historico.count()

    fraccion = cantidad / total

    return (
        df_historico
        .sample(
            withReplacement=False,
            fraction=fraccion,
            seed=seed
        )
        .limit(cantidad)
    )


# Cantidad de duplicados históricos por carga
cantidad_historicos = 100


# --------------------------------------------
# CARGA 002
# Recibe registros que ya llegaron en Carga 001
# --------------------------------------------

duplicados_002 = obtener_duplicados_historicos(
    carga_001,
    cantidad_historicos,
    SEED + 10
)

carga_002 = carga_002.unionByName(duplicados_002)


# --------------------------------------------
# CARGA 003
# Recibe registros de cargas anteriores
# --------------------------------------------

historico_003 = (
    carga_001
    .unionByName(carga_002)
)

duplicados_003 = obtener_duplicados_historicos(
    historico_003,
    cantidad_historicos,
    SEED + 11
)

carga_003 = carga_003.unionByName(duplicados_003)


# --------------------------------------------
# CARGA 004
# Recibe registros de todas las cargas anteriores
# --------------------------------------------

historico_004 = (
    carga_001
    .unionByName(carga_002)
    .unionByName(carga_003)
)

duplicados_004 = obtener_duplicados_historicos(
    historico_004,
    cantidad_historicos,
    SEED + 12
)

carga_004 = carga_004.unionByName(duplicados_004)


# --------------------------------------------
# RESULTADO
# --------------------------------------------

print(f"Carga 001: {carga_001.count()} registros")
print(f"Carga 002: {carga_002.count()} registros")
print(f"Carga 003: {carga_003.count()} registros")
print(f"Carga 004: {carga_004.count()} registros")

creacion de errores de data quality

In [0]:
# ============================================
# INYECTAR ERRORES DE CALIDAD
# ============================================

def inyectar_errores(df, porcentaje, seed):

    # Crear un identificador temporal por fila
    df = df.withColumn(
        "_id_fila",
        F.monotonically_increasing_id()
    )

    total = df.count()
    cantidad = int(total * porcentaje)

    # Seleccionar las filas que tendrán errores
    ids_error = (
        df
        .select("_id_fila")
        .orderBy(F.rand(seed))
        .limit(cantidad)
        .withColumn(
            "_tipo_error",
            F.pmod(F.abs(F.hash("_id_fila")), F.lit(5))
        )
    )

    # Unir el tipo de error
    df = df.join(
        ids_error,
        on="_id_fila",
        how="left"
    )

    # Aplicar errores
    df = (
        df

        # Error 0: fecha de recogida nula
        .withColumn(
            "tpep_pickup_datetime",
            F.when(
                F.col("_tipo_error") == 0,
                F.lit(None).cast("timestamp")
            ).otherwise(
                F.col("tpep_pickup_datetime")
            )
        )

        # Error 1: fecha de destino nula
        .withColumn(
            "tpep_dropoff_datetime",
            F.when(
                F.col("_tipo_error") == 1,
                F.lit(None).cast("timestamp")
            ).otherwise(
                F.col("tpep_dropoff_datetime")
            )
        )

        # Error 2: distancia negativa
        .withColumn(
            "trip_distance",
            F.when(
                F.col("_tipo_error") == 2,
                -F.abs(F.col("trip_distance"))
            ).otherwise(
                F.col("trip_distance")
            )
        )

        # Error 3: tarifa negativa
        .withColumn(
            "fare_amount",
            F.when(
                F.col("_tipo_error") == 3,
                -F.abs(F.col("fare_amount"))
            ).otherwise(
                F.col("fare_amount")
            )
        )

        # Error 4: destino anterior a recogida
        .withColumn(
            "tpep_dropoff_datetime",
            F.when(
                F.col("_tipo_error") == 4,
                F.col("tpep_pickup_datetime")
                - F.expr("INTERVAL 1 HOUR")
            ).otherwise(
                F.col("tpep_dropoff_datetime")
            )
        )

        .drop("_id_fila", "_tipo_error")
    )

    return df

In [0]:
# ============================================
# APLICAR ERRORES A LAS CARGAS
# ============================================

carga_001 = inyectar_errores(
    carga_001,
    PORCENTAJE_ERRORES,
    SEED + 20
)

carga_002 = inyectar_errores(
    carga_002,
    PORCENTAJE_ERRORES,
    SEED + 21
)

carga_003 = inyectar_errores(
    carga_003,
    PORCENTAJE_ERRORES,
    SEED + 22
)

carga_004 = inyectar_errores(
    carga_004,
    PORCENTAJE_ERRORES,
    SEED + 23
)

print(f"Carga 001: {carga_001.count()} registros")
print(f"Carga 002: {carga_002.count()} registros")
print(f"Carga 003: {carga_003.count()} registros")
print(f"Carga 004: {carga_004.count()} registros")

validad que existan los errores

In [0]:
# ============================================
# VALIDAR ERRORES INYECTADOS
# ============================================

for nombre, df in [
    ("Carga 001", carga_001),
    ("Carga 002", carga_002),
    ("Carga 003", carga_003),
    ("Carga 004", carga_004)
]:

    errores = (
        F.col("tpep_pickup_datetime").isNull()
        | F.col("tpep_dropoff_datetime").isNull()
        | (F.col("trip_distance") < 0)
        | (F.col("fare_amount") < 0)
        | (
            F.col("tpep_dropoff_datetime")
            < F.col("tpep_pickup_datetime")
        )
    )

    cantidad_errores = df.filter(errores).count()

    print(f"{nombre}: {cantidad_errores} registros con errores")

GUARDADO DE LA DATA RAW 

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.vol_raw_nyctaxi;

In [0]:
# ============================================
# RUTA DE ARCHIVOS RAW
# ============================================

ruta_raw = "/Volumes/workspace/default/vol_raw_nyctaxi"

print(ruta_raw)

In [0]:
# ============================================
# GUARDAR CARGAS SIMULADAS COMO CSV
# ============================================

carga_001.write.mode("overwrite").option("header", "true").csv(
    f"{ruta_raw}/carga_001"
)

carga_002.write.mode("overwrite").option("header", "true").csv(
    f"{ruta_raw}/carga_002"
)

carga_003.write.mode("overwrite").option("header", "true").csv(
    f"{ruta_raw}/carga_003"
)

carga_004.write.mode("overwrite").option("header", "true").csv(
    f"{ruta_raw}/carga_004"
)

print("Las cuatro cargas fueron guardadas correctamente.")

In [0]:
# ============================================
# VERIFICAR ARCHIVOS GENERADOS
# ============================================

display(dbutils.fs.ls(ruta_raw))

In [0]:
# ============================================
# VERIFICAR ARCHIVOS DE LA CARGA 001
# ============================================

display(
    dbutils.fs.ls(
        f"{ruta_raw}/carga_001"
    )
)

In [0]:
# ============================================
# LEER Y VALIDAR CARGA 001
# ============================================

df_prueba = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{ruta_raw}/carga_001")
)

print(f"Registros leídos: {df_prueba.count()}")

df_prueba.printSchema()

display(df_prueba.limit(5))